In [224]:
import pandas as pd
import numpy as np
import torch.nn as nn
import torch
from sklearn.preprocessing import StandardScaler

In [225]:
df=pd.read_csv('C:/a/ai/data/insurance.csv')
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [226]:
# 문자를 수로 변경 (원-핫 인코딩)
# 원-핫인코딩(yse일 때 true로 설정) 후 데이터타입 정수형으로 줌
df['target']=pd.get_dummies(df['smoker'])['yes'].astype('int')
df

,age,sex,bmi,children,smoker,region,charges,target
0,19,female,27.900,0,yes,southwest,16884.92400,1
1,18,male,33.770,1,no,southeast,1725.55230,0
2,28,male,33.000,3,no,southeast,4449.46200,0
3,33,male,22.705,0,no,northwest,21984.47061,0
4,32,male,28.880,0,no,northwest,3866.85520,0
...,...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830,0
1334,18,female,31.920,0,no,northeast,2205.98080,0
1335,18,female,36.850,0,no,southeast,1629.83350,0
1336,21,female,25.800,0,no,southwest,2007.94500,0


In [227]:
# smoker 열 삭제
df.drop('smoker', axis=1, inplace=True)
df

,age,sex,bmi,children,region,charges,target
0,19,female,27.900,0,southwest,16884.92400,1
1,18,male,33.770,1,southeast,1725.55230,0
2,28,male,33.000,3,southeast,4449.46200,0
3,33,male,22.705,0,northwest,21984.47061,0
4,32,male,28.880,0,northwest,3866.85520,0
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,northwest,10600.54830,0
1334,18,female,31.920,0,northeast,2205.98080,0
1335,18,female,36.850,0,southeast,1629.83350,0
1336,21,female,25.800,0,southwest,2007.94500,0


In [228]:
from sklearn.model_selection import train_test_split

features=df.iloc[:, 0:6]
label=df['target']

X_train, X_test, y_train, y_test=train_test_split(features, label, test_size=0.2, random_state=1)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(1070, 6) (268, 6) (1070,) (268,)


In [229]:
# 범주형 데이터들 원핫인코딩을 사용해 0,1(불린값으로 변환)
X_train_encode=pd.get_dummies(X_train)
X_test_encode=pd.get_dummies(X_test)

In [230]:
print(X_train_encode)
print(X_test_encode)

      age     bmi  children      charges  sex_female  sex_male  \
216    53  26.600         0  10355.64100        True     False   
731    53  21.400         1  10065.41300       False      True   
866    18  37.290         0   1141.44510       False      True   
202    60  24.035         0  13012.20865        True     False   
820    45  33.700         1   7445.91800       False      True   
...   ...     ...       ...          ...         ...       ...   
715    60  28.900         0  12146.97100       False      True   
905    26  29.355         2   4564.19145        True     False   
1096   51  34.960         2  44641.19740        True     False   
235    40  22.220         2  19444.26580        True     False   
1061   57  27.940         1  11554.22360       False      True   

      region_northeast  region_northwest  region_southeast  region_southwest  
216              False              True             False             False  
731              False             False         

In [231]:
# train에 있는 열이 test에 없을 수 있기 때문에 컬럼 순서를 train과 동일하게 맞추고, 없는 열은 0으로 다 채움
X_test_encode=X_test_encode.reindex(columns=X_test_encode.columns, fill_value=0)

In [232]:
X_test_encode.shape

(268, 10)

In [233]:
# X_train=pd.DataFrame({"region":["yongsan","jongno","mapo"]})
# X_test=pd.DataFrame({"region":["yongsan","jongno"]})

# pd.get_dummies(X_train) #3
# pd.get_dummies(X_test) #2

# X_test=X_test.reindex(columns=X_train.columns, fill_value=0)
# X_test

In [234]:
# 스케일링
sc=StandardScaler()
X_train_encode=sc.fit_transform(X_train_encode)
X_test_encode=sc.transform(X_test_encode)

In [235]:
# 넘파이배열 => 파이토치 텐서로 변환
# X_train_tensor=torch.FloatTensor(X_train_encode)
# X_test_tensor=torch.FloatTensor(X_test_encode)
X_train_tensor=torch.from_numpy(X_train_encode).float()
X_test_tensor=torch.from_numpy(X_test_encode).float()
print(X_train_tensor.shape, X_test_tensor.shape)

torch.Size([1070, 10]) torch.Size([268, 10])


In [236]:
# 테스트용도 텐서로 변환 -> 신경망 입력과 맞춘다(N,1)-> 차원추가
y_train_tensor=torch.from_numpy(y_train.values).float().unsqueeze(1)
y_test_tensor=torch.from_numpy(y_test.values).float().unsqueeze(1)
print(y_train_tensor.shape, y_test_tensor.shape)

torch.Size([1070, 1]) torch.Size([268, 1])


In [237]:
class SmokeModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear=nn.Sequential(
            nn.Linear(input_dim, 200),
            # 음수영역에도 작은 기울기를 주면서 죽은 신호 완화시킴
            nn.LeakyReLU(0.1),
            nn.Linear(200,100),
            nn.Dropout(0.3),
            nn.LeakyReLU(0.1),
            nn.Linear(100,20),
            nn.Dropout(0.3),
            nn.LeakyReLU(0.1),
            nn.Linear(20,5),
            nn.Dropout(0.3),
            nn.LeakyReLU(0.1),
            nn.Linear(5, output_dim), # 점점 입력차원수를 줄여가면서 차원 축소함
            nn.Sigmoid() # 이진 분류 시 필수 메소드
        )

    # 모델이 입력받으면 forward가 호출된다
    def forward(self, x):
        y=self.linear(x)
        return y

In [238]:
input_dim=X_train_tensor.shape[1]
output_dim=y_train_tensor.shape[1]

In [239]:
model=SmokeModel(input_dim, output_dim)

In [240]:
# 손실함수 정의
bceloss=nn.BCELoss()
optimizer=torch.optim.Adam(model.parameters(), lr=0.001)
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [241]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset=TensorDataset(X_train_tensor, y_train_tensor)
train_loader=DataLoader(train_dataset, batch_size=250, shuffle=True)


loss1=[]
num_epochs=100

for epoch in range(num_epochs):
    for x,y in train_loader:
        y_pred=model(x)
        loss=bceloss(y_pred,y)
        loss1.append(loss)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    
    print(sum(loss1)/len(loss1))
    

tensor(0.6339, grad_fn=<DivBackward0>)
tensor(0.6223, grad_fn=<DivBackward0>)
tensor(0.6020, grad_fn=<DivBackward0>)
tensor(0.5785, grad_fn=<DivBackward0>)
tensor(0.5546, grad_fn=<DivBackward0>)
tensor(0.5299, grad_fn=<DivBackward0>)
tensor(0.5039, grad_fn=<DivBackward0>)
tensor(0.4829, grad_fn=<DivBackward0>)
tensor(0.4612, grad_fn=<DivBackward0>)
tensor(0.4398, grad_fn=<DivBackward0>)
tensor(0.4220, grad_fn=<DivBackward0>)
tensor(0.4037, grad_fn=<DivBackward0>)
tensor(0.3873, grad_fn=<DivBackward0>)
tensor(0.3723, grad_fn=<DivBackward0>)
tensor(0.3576, grad_fn=<DivBackward0>)
tensor(0.3447, grad_fn=<DivBackward0>)
tensor(0.3337, grad_fn=<DivBackward0>)
tensor(0.3229, grad_fn=<DivBackward0>)
tensor(0.3139, grad_fn=<DivBackward0>)
tensor(0.3049, grad_fn=<DivBackward0>)
tensor(0.2968, grad_fn=<DivBackward0>)
tensor(0.2889, grad_fn=<DivBackward0>)
tensor(0.2818, grad_fn=<DivBackward0>)
tensor(0.2753, grad_fn=<DivBackward0>)
tensor(0.2692, grad_fn=<DivBackward0>)
tensor(0.2643, grad_fn=<D

In [242]:
total=0
correct=0

# 역전파 비활성화 => Gradient 계산 끄기(메모리 절약)
with torch.no_grad():
    for x,y in train_loader:
        outputs=model(x)
        predict=(outputs > 0.5).float() # 이진 예측
        total+=y.size(0)
        correct+=(predict==y).sum().item()
        print(y.size(0))
        
accuracy=correct/total
print(sum(loss1)/len(loss1))
print(accuracy)
# 정확도 100% 이고, 손실은 0.0446 => 확신 아직 100%로는 아닌상태

250
250
250
250
70
tensor(0.1348, grad_fn=<DivBackward0>)
0.9775700934579439


In [243]:
model

SmokeModel(
  (linear): Sequential(
    (0): Linear(in_features=10, out_features=200, bias=True)
    (1): LeakyReLU(negative_slope=0.1)
    (2): Linear(in_features=200, out_features=100, bias=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): LeakyReLU(negative_slope=0.1)
    (5): Linear(in_features=100, out_features=20, bias=True)
    (6): Dropout(p=0.3, inplace=False)
    (7): LeakyReLU(negative_slope=0.1)
    (8): Linear(in_features=20, out_features=5, bias=True)
    (9): Dropout(p=0.3, inplace=False)
    (10): LeakyReLU(negative_slope=0.1)
    (11): Linear(in_features=5, out_features=1, bias=True)
    (12): Sigmoid()
  )
)

In [244]:
y_test_tensor.shape

torch.Size([268, 1])

In [245]:
print(X_test_tensor.shape, y_test_tensor.shape)

torch.Size([268, 10]) torch.Size([268, 1])


In [246]:
# 테스트용 데이터셋
from torch.utils.data import TensorDataset, DataLoader

test_dataset=TensorDataset(X_test_tensor, y_test_tensor)
test_loader=DataLoader(test_dataset, batch_size=250, shuffle=False)


y_pred=[]

# 평가용 -> 뉴런 연결 끊는다.(dropout)
model.eval()
with torch.no_grad():
    for x,_ in test_loader:
        y_test_pred=model(x)
        y_test_pred_sigmoid=torch.round(y_test_pred)
        y_pred.extend(y_test_pred_sigmoid)

print(y_pred)
y_pred_tensor=torch.tensor(y_pred)

[tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([1.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([1.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([1.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([1.]), tensor([1.]), tensor([1.]), tensor([0.]), tensor([1.]), tensor([1.]), tensor([0.]), tensor([1.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([1.]), tensor([1.]), tensor([1.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([1.]), tensor([0.]), tensor([1.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([1.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([0.]), tensor([1.]), tensor([1.]), tenso

In [247]:
from sklearn.metrics import f1_score, precision_score

In [248]:
# 텐서 -> 넘파이로 바꿔서 성능평가
y_numpy_test=y_test_tensor.numpy()
y_numpy_test=y_pred_tensor.numpy()

In [249]:
# 중복원소 제거하고 고유한 값 확인
# 이진분류 -> 정상적인 분류로 예측함
torch.unique(y_pred_tensor)

tensor([0., 1.])

In [250]:
torch.unique(y_test_tensor)

tensor([0., 1.])

In [223]:
torch.unique(y_test_tensor, return_counts=True)

(tensor([0., 1.]), tensor([214,  54]))